# EXP-013 원인 분석: 질의 재작성은 왜 순위를 내렸나

EXP-013은 기각(dev-full Recall@5 −0.50%p [−0.89, −0.10])이다. 재작성 대상 362개 질의에서 무엇이 순위를 올리고 내렸는지 본다. EXP 문서 "분석" 절의 요소별 기여와 설계 우려 두 가지(재작성이 BM25 쪽에 맞는 형태인지, 재작성 목록 두 개가 같은 방향으로 표를 더해 방해 문서를 올리는지)를 확인하고, 개선책 후보를 같은 질의에 다시 계산한다. 개선책 계산은 판정에 쓰지 않는다. 같은 dev-full에서 고른 변형을 다시 dev-full로 판정하면 효과가 높게 나오므로, 다음 변형을 고르는 참고로만 쓴다.

In [1]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd

from rag.conditions import build_extractor
from rag.eval.metrics import gold_ranks, load_gold, read_run
from rag.index import encoder, load_cfg, load_model
from rag.retrieve import Retriever, conditions_first, rrf_scores
from rag.rewrite import parse
from rag.search import embed_queries, query_embeddings, query_files, query_list, read_texts

C:\workspace\meet-rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ROOT = Path.cwd().parent
RUNS = ROOT / 'data' / 'runs'
BASE_RUN = 'snowflake-ko-fixed-512-64-hybrid-cond-filter-law-ms256-dev-full.csv'
RW_RUN = 'snowflake-ko-fixed-512-64-hybrid-cond-filter-law-ms256-rw-dev-full.csv'
REWRITES = ROOT / 'data' / 'rewrites' / 'openrouter-rw_v1.json'
TOP = 5
N_SHOW = 10
N_TIMING = 50

## 재작성 대상과 결과

In [3]:
cfg = load_cfg(['+exp=exp013', 'index.scope=full'])
gold = load_gold(cfg, 'dev-full')
base, rw = read_run(RUNS / BASE_RUN), read_run(RUNS / RW_RUN)
lists_of = {n: r.groupby('qid')['doc_id'].apply(list) for n, r in [('base', base), ('rw', rw)]}
gated = [q for q in lists_of['base'].index if lists_of['base'][q] != lists_of['rw'][q]]
rb, rr = gold_ranks(base, gold, 10), gold_ranks(rw, gold, 10)
hit_b, hit_r = rb[gated] <= TOP, rr[gated] <= TOP
outcome = pd.Series('same', index=gated)
outcome[~hit_b & hit_r] = 'gain'
outcome[hit_b & ~hit_r] = 'loss'
print('순위가 바뀐 질의', len(gated))
print(outcome.value_counts().to_dict())
print('재작성 대상 중 기준 top5 성공', int(hit_b.sum()), '/', len(gated))

순위가 바뀐 질의 362
{'same': 327, 'loss': 25, 'gain': 10}
재작성 대상 중 기준 top5 성공 231 / 362


재작성 대상은 1차 검색 신뢰도가 낮은 질의지만, 그중 다수는 기준이 이미 상위 5위에 넣고 있었다. 잃을 수 있는 질의가 얻을 수 있는 질의보다 많은 구조다.

## 목록별 정답 순위 (원 질의 vs 재작성 질의)

In [4]:
ret = Retriever(cfg)
qids, text_path = query_list(cfg)
texts = dict(zip(qids, read_texts(text_path, qids), strict=True))
q_all = query_embeddings(cfg, qids, text_path, query_files(cfg)['emb'])
q_emb = dict(zip(qids, q_all, strict=True))
ex = build_extractor(cfg)
items = json.loads(REWRITES.read_text(encoding='utf-8'))['items']
rw_text = {q: parse(items[q]['raw'], cfg.retriever.rewrite.max_chars) for q in gated}
start = time.perf_counter()
rw_emb = dict(zip(gated, embed_queries(cfg, [rw_text[q] for q in gated]), strict=True))
print('재작성 질의 임베딩', len(gated), '건', round(time.perf_counter() - start, 1), '초')

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 2161.06it/s]


재작성 질의 임베딩 362 건 84.5 초


In [5]:
tok = dict(zip(gated, ret.query_tokens([texts[q] for q in gated]), strict=True))
rw_tok = dict(zip(gated, ret.query_tokens([rw_text[q] for q in gated]), strict=True))
named, matched = {}, {}
for q in gated:
    row = ret.chunk_emb @ q_emb[q]
    named[q], matched[q], _ = ret.lists(row, tok[q], ex.extract(texts[q]),
                                        rewrite=(ret.chunk_emb @ rw_emb[q], rw_tok[q]))


def gold_rank(lst, q):
    g = set(gold.loc[[q], 'doc_id'])
    return next((i for i, d in enumerate(lst, 1) if d in g), np.nan)


lr = pd.DataFrame({q: {n: gold_rank(lst, q) for n, lst in named[q].items()} for q in gated}).T
lr['outcome'] = outcome
print(lr.groupby('outcome')[['dense', 'dense_rw', 'bm25', 'bm25_rw']].median())

         dense  dense_rw  bm25  bm25_rw
outcome                                
gain      12.0       4.5  11.0      4.0
loss       6.0      23.5   2.0     17.0
same       3.0       6.0   2.0      2.0


### 요소별 기여 (설계 우려 1)

재작성 목록이 정답을 원 목록보다 위에 둔 질의 수. 목록 깊이(100) 밖은 101로 본다.

In [6]:
f = lr.fillna(101)
rows = []
for kind in ['dense', 'bm25']:
    for o in ['gain', 'loss', 'same', 'all']:
        s = f if o == 'all' else f[f['outcome'] == o]
        rows.append({'목록': kind, 'outcome': o, 'n': len(s),
                     '재작성이 위': int((s[f'{kind}_rw'] < s[kind]).sum()),
                     '재작성이 아래': int((s[f'{kind}_rw'] > s[kind]).sum()),
                     '재작성 top5': int((s[f'{kind}_rw'] <= TOP).sum()),
                     '원 top5': int((s[kind] <= TOP).sum())})
print(pd.DataFrame(rows).to_string(index=False))

   목록 outcome   n  재작성이 위  재작성이 아래  재작성 top5  원 top5
dense    gain  10       8        1         6       1
dense    loss  25       1       24         2      10
dense    same 327      63      157       117     165
dense     all 362      72      182       125     176
 bm25    gain  10       8        2         5       3
 bm25    loss  25       1       22         4      15
 bm25    same 327      48      146       160     191
 bm25     all 362      57      170       169     209


### 정답을 밀어낸 문서 (설계 우려 2)

밀린 질의마다 재작성 뒤 상위 5위에 새로 들어온 문서가 재작성 목록 두 개(dense_rw, bm25_rw)의 상위 10위에 함께 있었는지 센다.

In [7]:
rows = []
for q in outcome[outcome == 'loss'].index:
    new = [d for d in lists_of['rw'][q][:TOP] if d not in lists_of['base'][q][:TOP]]
    for d in new:
        pos = {n: (named[q][n].index(d) + 1 if d in named[q][n] else None) for n in named[q]}
        rows.append({'qid': q, 'doc': d, **pos})
pushed = pd.DataFrame(rows)
both = pushed['dense_rw'].le(10) & pushed['bm25_rw'].le(10)
print('밀린 질의', pushed['qid'].nunique(), ' 새로 들어온 문서', len(pushed))
only_d = pushed['dense_rw'].le(10) & ~both
only_b = pushed['bm25_rw'].le(10) & ~both
orig_top = pushed['dense'].le(10) | pushed['bm25'].le(10)
print('재작성 목록 둘 다 top10', int(both.sum()), ' dense_rw만', int(only_d.sum()),
      ' bm25_rw만', int(only_b.sum()))
print('원 목록(dense·bm25) 중 하나라도 top10', int(orig_top.sum()))

밀린 질의 25  새로 들어온 문서 57
재작성 목록 둘 다 top10 14  dense_rw만 17  bm25_rw만 18
원 목록(dense·bm25) 중 하나라도 top10 33


## 밀린 질의 예시

질의 원문과 재작성(앞 N_SHOW개). 재작성이 대상·뜻을 바꿨는지 본다.

In [8]:
for q in outcome[outcome == 'loss'].index[:N_SHOW]:
    print(f'- [{q}] 기준 {rb[q]:.0f}위 → {rr[q]}위')
    print('  원 ', texts[q].strip())
    print('  재작성', rw_text[q])

- [043711-0001] 기준 1위 → nan위
  원  다른 주제와 관련하여 용인할 수 있는 대안은 무엇이며, 다른 위원들과 협력하여 진행될 예정인가요?
  재작성 대안 마련 및 위원 간 협력 방안 추진 계획
- [046280-0025] 기준 2위 → 8.0위
  원  대통령이 직접 나서서 장기적인 대한민국의 인구문제 대책을 수립하고 예산문제를 해결하는 방안에 대한 검토와 협의를 어떻게 진행하고 있으며, 이러한 제안을 구체화하는 계획은 어떤가요?
  재작성 대통령 주도 저출산 고령화 인구문제 대책 수립 예산 확보 방안 검토 및 협의 계획
- [046592-0011] 기준 1위 → 6.0위
  원  해군 기지 건설 사업에 대한 찬반 논쟁이 극심했던 당시의 정치적·사회적 분위기와 갈등 상황에 대한 귀하의 관찰과 생각을 공유해 주십시오.
  재작성 해군 기지 건설 사업 찬반 논쟁 정치적 사회적 분위기 갈등 상황
- [046608-0007] 기준 3위 → nan위
  원  에너지 기본계획이나 전력수급계획이 예기치 못한 원전 중지나 석탄발전 중단에 대비하는 데에는 어떤 한계가 있었던 것으로 보시나요?
  재작성 에너지기본계획 전력수급기본계획 원전 중지 석탄발전 중단 대비 한계점 및 문제점
- [046680-0016] 기준 1위 → nan위
  원  다양한 정당에서 단체장들이 대선 출마에 희망을 품고 있습니다. 이에 대한 당내 경선 시 효율적인 인력 조성을 위해 정당 간 합의는 가능할 것으로 보이나요?
  재작성 정당 단체장 대통령 선거(대선) 출마 정당 간 합의 당내 경선 인력 운용 효율화
- [046969-0002] 기준 3위 → 10.0위
  원  중소기업이 기술적 탈취로부터 보호받지 못해 어려움을 겪는데, 정부가 강력한 입장을 취해주거나 법률적 배경을 튼튼하게 해줄 계획이 있을까요?
  재작성 중소기업 기술탈취 보호 대책 정부 대응 방안 및 법적 근거 강화 계획
- [047620-0017] 기준 1위 → 8.0위
  원  5억 원의 설계예산 증액이 기본계획에

In [9]:
for q in outcome[outcome == 'gain'].index:
    print(f'+ [{q}] 기준 {rb[q]}위 → {rr[q]:.0f}위')
    print('  원 ', texts[q].strip())
    print('  재작성', rw_text[q])

+ [047217-0002] 기준 7.0위 → 2위
  원  이 상황에서 정부가 이를 사실상 단속할 수 있는 여지는 있을까요?
  재작성 정부 단속 가능성 및 실효성 여부
+ [047495-0018] 기준 nan위 → 5위
  원  경제 성장과 관련하여 국내외의 성공적인 사례를 들며, 특히 독일의 하르츠 개혁 및 불란서의 경제정책을 언급하셨습니다. 현재의 혁신성장 방향과 정책이 충분한 생산성 향상과 경제 성장을 이루어낼 수 있는 방향인지에 대한 자신의 입장을 간단히 말씀해 주십시오.
  재작성 경제 성장 독일 하르츠 개혁 프랑스(불란서) 경제정책 혁신성장 방향 생산성 향상
+ [048620-0016] 기준 nan위 → 3위
  원  대통령의 주요 관심사는 비핵화에 있는 것으로 이해해도 되는 건가요?
  재작성 대통령 비핵화 관심사 및 추진 방향
+ [048643-0016] 기준 6.0위 → 2위
  원  '국제협력 강화'라는 목표를 달았지만, 현재 국제협력 사업은 북한을 제외하고 다른 국가와 협력하는 것인가요?
  재작성 국제협력 강화 사업 대상 국가 북한 제외 여부 및 타국 협력 현황
+ [049590-0007] 기준 8.0위 → 4위
  원  최근에 KBS가 이슈가 된 사건들에 대해 여론에서는 편파 보도와 방송 외압 논란 등이 제기되고 있습니다. 이에 대한 KBS의 입장은 어떠한가요?
  재작성 한국방송공사(KBS) 편파 보도 방송 외압 논란 입장 및 대응
+ [049590-0015] 기준 8.0위 → 2위
  원  보도를 안 보셔서 모르시다고 했는데, 그럼 그 자리에 앉아 계실 이유가 없지 않나요?
  재작성 보도 내용 인지 여부 및 직무 수행 적절성 책임론
+ [049652-0010] 기준 9.0위 → 1위
  원  자동비양장치를 도입하기로 결정한 이유는 무엇인가요?
  재작성 자동비양장치 도입 결정 사유 및 필요성
+ [049939-0009] 기준 8.0위 → 5위
  원  마스크 수출 및 국내 공급에 대한 현재 상황을 감안할 때, 정부의 마스크 관리

## 개선책 후보의 사후 계산 (판정 아님)

같은 362개 질의에서 결합 방식만 바꿔 다시 순위를 낸다. 재작성하지 않은 질의는 기준과 같으므로 차이는 이 질의들에서만 생긴다.

In [10]:
def rank_with(q, use, weights):
    names = [n for n in named[q] if n in use]
    score = rrf_scores([named[q][n] for n in names], cfg.retriever.fusion.rrf_k,
                       [weights.get(n, 1.0) for n in names])
    fused = sorted(score, key=score.__getitem__, reverse=True)
    return conditions_first(fused, matched[q])[: cfg.retriever.top_k]


orig = ['dense', 'bm25', 'conditions']
variants = {
    'EXP-013 (dense_rw + bm25_rw)': (orig + ['dense_rw', 'bm25_rw'], {}),
    'bm25_rw만': (orig + ['bm25_rw'], {}),
    'dense_rw만': (orig + ['dense_rw'], {}),
    '둘 다 가중치 0.5': (orig + ['dense_rw', 'bm25_rw'], {'dense_rw': 0.5, 'bm25_rw': 0.5}),
}
rows = []
for name, (use, w) in variants.items():
    r = pd.Series({q: gold_rank(rank_with(q, use, w), q) for q in gated})
    hit = r <= TOP
    rows.append({'변형': name, '진입': int((~hit_b & hit).sum()), '이탈': int((hit_b & ~hit).sum()),
                 '순증': int(hit.sum() - hit_b.sum()),
                 'Recall@5 차이(%p)': round((hit.sum() - hit_b.sum()) / len(gold) * 100, 2)})
print(pd.DataFrame(rows).to_string(index=False))

                          변형  진입  이탈  순증  Recall@5 차이(%p)
EXP-013 (dense_rw + bm25_rw)  10  25 -15            -0.50
                    bm25_rw만  23  17   6             0.20
                   dense_rw만   8  22 -14            -0.46
                 둘 다 가중치 0.5   7  14  -7            -0.23


첫 줄(EXP-013)이 판정 수치(오름 10, 내림 25, −0.50%p)와 같으면 이 계산이 실제 검색과 같은 결합을 재현한 것이다.

## 지연 (재작성 대상 질의 하나)

In [11]:
secs = pd.Series({q: items[q]['seconds'] for q in gated})
print('OpenRouter 재작성 호출(초)', secs.describe(percentiles=[0.5, 0.95]).round(2).to_dict())
model = load_model(cfg, cfg.embedding.query_max_tokens)
encode = encoder(model, cfg, cfg.embedding.get('query_prompt'))
sample = gated[:N_TIMING]
t = {'원 질의만': [], '재작성 목록 포함': [], '재작성 질의 임베딩': []}
for q in sample:
    s = time.perf_counter()
    ret.candidates(ret.chunk_emb @ q_emb[q], tok[q], ex.extract(texts[q]))
    t['원 질의만'].append(time.perf_counter() - s)
    s = time.perf_counter()
    e = encode([rw_text[q]])[0]
    t['재작성 질의 임베딩'].append(time.perf_counter() - s)
    s = time.perf_counter()
    ret.candidates(ret.chunk_emb @ q_emb[q], tok[q], ex.extract(texts[q]),
                   rewrite=(ret.chunk_emb @ e, ret.query_tokens([rw_text[q]])[0]))
    t['재작성 목록 포함'].append(time.perf_counter() - s)
for k, v in t.items():
    print(k, f'p50 {np.median(v) * 1000:.0f}ms, p95 {np.percentile(v, 95) * 1000:.0f}ms')

OpenRouter 재작성 호출(초) {'count': 362.0, 'mean': 1.02, 'std': 0.57, 'min': 0.45, '50%': 0.81, '95%': 2.28, 'max': 4.11}


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 2219.55it/s]


원 질의만 p50 90ms, p95 157ms
재작성 목록 포함 p50 231ms, p95 412ms
재작성 질의 임베딩 p50 177ms, p95 307ms


운영 지연은 재작성 대상 질의에만 재작성 호출 + 재작성 질의 인코딩(모델은 한 번 읽어 둠) + 검색 차이가 더해진다.

## 정리

- 선정: 재작성 대상 362개 중 231개(64%)는 기준이 이미 상위 5위에 정답을 넣고 있었다. 내려간 25개는 모두 여기서 나왔다. 판단 신호(A′, B′)가 "기준이 놓칠 질의"를 잘 고르지 못했다
- 재작성 자체: 재작성 질의만으로 검색하면 정답 상위 5위가 원 질의보다 적다(dense 176 → 125, BM25 209 → 169). 기준이 놓친 131개 중 살린 것은 10개(7.6%)다. 놓친 질의에는 대상이 질의에 없는 경우("그 사업 예산")가 많아, 질의만 보고 다시 쓰는 방식으로는 채울 정보가 없다
- 설계 우려 1(BM25에 맞는 형태): 맞다. bm25_rw만 더하면 순증 +6(+0.20%p), dense_rw만 더하면 −14(−0.46%p)
- 설계 우려 2(두 목록이 같은 방향으로 표를 더함): 일부만 맞다. 정답을 밀어낸 57개 중 재작성 목록 둘 다 상위 10위는 14개다
- 어느 결합 변형도 +1.0%p에 닿지 않는다(가장 나은 bm25_rw만 +0.20%p)
- 지연(재작성 대상 질의 하나, OpenRouter): 재작성 호출 p50 0.81초·p95 2.28초, 재작성 질의 인코딩 p50 177ms, 검색 90 → 231ms. 합해 p50 약 1.1초가 더해진다. 배포 경로(Google 무료 쿼터)는 dev-small 10건 평균 46초다